<a href="https://colab.research.google.com/github/vthkrl/big_data_fall26/blob/main/Big_Data_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



1. Data consolidation: combine information from a wide range of data sources to build a comprehensive picture of each NYC community district.




Notes on datasets:
- Facility info is in `facilities.csv`, the `cd` column lists the district number
- Data on crime rate is in `neighborhood_indicators.xlsx`, and needs to be extracted (`crime_all_rt` column)
- Median income is in `median-income.csv` with the *name* of the CD in the `location_name` column (name can be mapped to district number using the neighborhood indicators table)
- Remaining metrics each have their own csv file, each of which has district number in the `GeoID` column
- Some metrics are grouped by individual year while others are in four-year blocks. To account for this, we could potentially take the single-year data and average every block of four years to create corresponding block values.

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from google.colab import drive
drive.mount('/content/drive')
folder_path = "/content/drive/MyDrive/Project/datasets/"

spark = SparkSession.builder \
    .appName("FacilityPredictor") \
    .getOrCreate()

facilities_path = folder_path + "facilities.csv"
facilities = spark.read.csv(facilities_path, header=True, inferSchema=True)

neighborhood_indicators_path = folder_path + "neighborhood_indicators.xlsx"
neighborhood_indicators = spark.read.csv(neighborhood_indicators_path, header=True, inferSchema=True)

# metrics
# 1. rent burden, done
rent_burden_rate_path = folder_path + "rent_burden.csv"
rent_burden_rate = spark.read.csv(rent_burden_rate_path, header=True, inferSchema=True)

# 2. crowding, done
crowding_rate_path = folder_path + "crowding.csv"
crowding_rate = spark.read.csv(crowding_rate_path, header=True, inferSchema=True)

# 3. median income, done
median_income_path = folder_path + "median_income_cds.csv"
median_income = spark.read.csv(median_income_path, header=True, inferSchema=True)

# 4. poverty, done
poverty_rate_path = folder_path + "poverty.csv"
poverty_rate = spark.read.csv(poverty_rate_path, header=True, inferSchema=True)

# 5. crime, done
crime_rate_path = folder_path + "crime_rate_parsed.csv"
crime_rate = spark.read.csv(crime_rate_path, header=True, inferSchema=True)

# 6. high school graduation, done
high_school_graduation_rate_path = folder_path + "high_school.csv"
high_school_graduation_rate = spark.read.csv(high_school_graduation_rate_path, header=True, inferSchema=True)

# 7. fine particles, done
fine_particles_concentration_path = folder_path + "fine_particles.csv"
fine_particles_concentration = spark.read.csv(fine_particles_concentration_path, header=True, inferSchema=True)

# 8. trash cans, done
trash_can_density_path = folder_path + "trash_cans.csv"
trash_can_density = spark.read.csv(trash_can_density_path, header=True, inferSchema=True)

# 9. premature mortality, done
premature_mortality_rate_path = folder_path + "mortality.csv"
premature_mortality_rate = spark.read.csv(premature_mortality_rate_path, header=True, inferSchema=True)

# 10. independency, done
independency_rate_path = folder_path + "indep_living.csv"
independency_rate = spark.read.csv(independency_rate_path, header=True, inferSchema=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Data cleaning: standardize date ranges and spatial units from each data source, handle facilities with poor or absent location information.


In [2]:
# community districts
cds = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 501, 502, 503]
cds_tuples = [(cd,) for cd in cds]
scores_start = spark.createDataFrame(cds_tuples, schema=["cd"])

# facilities
facility_counts = (
    facilities
    .groupBy("cd")
    .pivot("facgroup")
    .count()
    .na.fill(0)
)

In [3]:
# raw scores

scores = scores_start.join(
    facility_counts,
    on="cd",
    how="left"
)

# clean health data

# rent burden
rent_burden_rate_2015_19 = rent_burden_rate.filter(rent_burden_rate["TimePeriod"].cast('string') == "2015-19")
scores = scores.join(
    rent_burden_rate_2015_19.select(
        rent_burden_rate_2015_19["GeoID"].alias("cd"),
        F.col("Percent").alias("Rent burden rate, 2015-2019"),
    ),
    on="cd",
    how="left"
)

# crowding
crowding_rate_2015_19 = crowding_rate.filter(crowding_rate["TimePeriod"].cast("string") == "2015-19")
scores = scores.join(
    crowding_rate_2015_19.select(
        crowding_rate_2015_19["GeoID"].alias("cd"),
        F.col("Percent").alias("Crowding rate, 2015-2019"),
    ),
    on="cd",
    how="left"
)

# median income
median_income_2015_19 = median_income.filter(median_income["timeframe"].cast("string") == "2017")
scores = scores.join(
    median_income_2015_19.select(
        median_income_2015_19["region_id"].alias("cd"),
        F.col("data").alias("Median income, 2017"),
    ),
    on="cd",
    how="left"
)

# poverty
poverty_rate_2015_19 = poverty_rate.filter(poverty_rate["TimePeriod"].cast("string") == "2015-19")
scores = scores.join(
    poverty_rate_2015_19.select(
        poverty_rate_2015_19["GeoID"].alias("cd"),
        F.col("Percent").alias("Poverty rate, 2015-2019"),
    ),
    on="cd",
    how="left"
)

# crime
crime_rate_2015_19 = crime_rate.filter(crime_rate["year_block"].cast("string") == "2015-19")
scores = scores.join(
    crime_rate_2015_19.select(
        crime_rate_2015_19["region_id"].alias("cd"),
        F.col("avg_crime_rt").alias("Crime rate, 2015-19"),
    ),
    on="cd",
    how="left"
)

# high school graduation
high_school_graduation_rate_2015_19 = high_school_graduation_rate.filter(high_school_graduation_rate["TimePeriod"].cast("string") == "2015-19")
scores = scores.join(
    high_school_graduation_rate_2015_19.select(
        high_school_graduation_rate_2015_19["GeoID"].alias("cd"),
        F.col("Percent").alias("High school graduation rate, 2015-19"),
    ),
    on="cd",
    how="left"
)

# fine particles
fine_particles_concentration_2017 = fine_particles_concentration.filter(fine_particles_concentration["TimePeriod"].cast("string") == "2017")
scores = scores.join(
    fine_particles_concentration_2017.select(
        fine_particles_concentration_2017["GeoID"].alias("cd"),
        F.col("Annual mean mcg/m3").alias("Fine particles concentration, 2017"),
    ),
    on="cd",
    how="left"
)

# trash cans
trash_can_density_2022 = trash_can_density
scores = scores.join(
    trash_can_density_2022.select(
        trash_can_density_2022["GeoId"].alias("cd"),
        F.col("Density per square mile").alias("Trash can density per square mile, 2022"),
    ),
    on="cd",
    how="left"
)

# premature mortality
premature_mortality_rate_2017 = premature_mortality_rate.filter(premature_mortality_rate["TimePeriod"].cast("string") == "2017")
scores = scores.join(
    premature_mortality_rate_2017.select(
        premature_mortality_rate_2017["GeoID"].alias("cd"),
        ((F.col("Age-adjusted rate per 100,000")) / 1000).alias("Premature mortality rate, 2017"),
    ),
    on="cd",
    how="left"
)

# independency
independency_rate_2015_19 = independency_rate.filter(independency_rate["TimePeriod"].cast("string") == "2015-19")
scores = scores.join(
    independency_rate_2015_19.select(
        independency_rate_2015_19["GeoID"].alias("cd"),
        F.col("Percent").alias("Independency rate, 2015-2019"),
    ),
    on="cd",
    how="left"
)

# show the scores!
scores.show()

+---+--------------+-----+--------------------------+---------------------------------------------+---------------------+-----------------------------+------------------+-----------+----------------+----------------+--------------+-----------------------+---------+-----------------------------+------------------------------+--------------+----------------+-------------+--------------+-----------+------------------+--------------+----------------------------------+--------------------+--------------+---------------------------+------------------------+-------------------+-----------------------+-------------------+------------------------------------+----------------------------------+---------------------------------------+------------------------------+----------------------------+
| cd|ADULT SERVICES|CAMPS|CHILD SERVICES AND WELFARE|CITY AGENCY PARKING, MAINTENANCE, AND STORAGE|CULTURAL INSTITUTIONS|DAY CARE AND PRE-KINDERGARTEN|EMERGENCY SERVICES|HEALTH CARE|HIGHER EDUCATION|HISTORI

In [4]:
# percentiles

from pyspark.sql import Window

rent_window = Window.orderBy("Rent burden rate, 2015-2019")
percentiles = scores.withColumn("Rent burden percentile, 2015-2019", F.percent_rank().over(rent_window))

crowding_window = Window.orderBy("Crowding rate, 2015-2019")
percentiles = percentiles.withColumn("Crowding percentile, 2015-2019", F.percent_rank().over(crowding_window))

income_window = Window.orderBy("Median income, 2017")
percentiles = percentiles.withColumn("Median income percentile, 2017", F.percent_rank().over(income_window))

poverty_window = Window.orderBy("Poverty rate, 2015-2019")
percentiles = percentiles.withColumn("Poverty percentile, 2015-2019", F.percent_rank().over(poverty_window))

crime_window = Window.orderBy("Crime rate, 2015-19")
percentiles = percentiles.withColumn("Crime percentile, 2015-19", F.percent_rank().over(crime_window))

high_school_window = Window.orderBy("High school graduation rate, 2015-19")
percentiles = percentiles.withColumn("High school graduation percentile, 2015-19", F.percent_rank().over(high_school_window))

fine_particles_window = Window.orderBy("Fine particles concentration, 2017")
percentiles = percentiles.withColumn("Fine particles percentile, 2017", F.percent_rank().over(fine_particles_window))

trash_can_window = Window.orderBy("Trash can density per square mile, 2022")
percentiles = percentiles.withColumn("Trash can percentile, 2022", F.percent_rank().over(trash_can_window))

premature_mortality_window = Window.orderBy("Premature mortality rate, 2017")
percentiles = percentiles.withColumn("Premature mortality percentile, 2017", F.percent_rank().over(premature_mortality_window))

independency_window = Window.orderBy("Independency rate, 2015-2019")
percentiles = percentiles.withColumn("Independency percentile, 2015-2019", F.percent_rank().over(independency_window))

# show the percentiles!
# percentiles.show()

In [5]:
w1 = -1
w2 = -1
w3 = 5
w4 = -2
w5 = -1
w6 = 4
w7 = -0.5
w8 = 1
w9 = -1
w10 = 3

positives = (num for num in (w1, w2, w3, w4, w5, w6, w7, w8, w9, w10) if num > 0)
negatives = (num for num in (w1, w2, w3, w4, w5, w6, w7, w8, w9, w10) if num < 0)

sum_positives = sum(positives)
sum_negatives = -1 * sum(negatives)

def normalize(var, sum_positives, sum_negatives):
  if var >= 0:
    return var / sum_positives
  else:
    return var / sum_negatives

w1 = normalize(w1, sum_positives, sum_negatives)
w2 = normalize(w2, sum_positives, sum_negatives)
w3 = normalize(w3, sum_positives, sum_negatives)
w4 = normalize(w4, sum_positives, sum_negatives)
w5 = normalize(w5, sum_positives, sum_negatives)
w6 = normalize(w6, sum_positives, sum_negatives)
w7 = normalize(w7, sum_positives, sum_negatives)
w8 = normalize(w8, sum_positives, sum_negatives)
w9 = normalize(w9, sum_positives, sum_negatives)
w10 = normalize(w10, sum_positives, sum_negatives)

overall_percentile = percentiles.withColumn(
    "Overall percentile",
      w1 * F.col("Rent burden percentile, 2015-2019")
    + w2 * F.col("Crowding percentile, 2015-2019")
    + w3 * F.col("Median income percentile, 2017")
    + w4 * F.col("Poverty percentile, 2015-2019")
    + w5 * F.col("Crime percentile, 2015-19")
    + w6 * F.col("High school graduation percentile, 2015-19")
    + w7 * F.col("Fine particles percentile, 2017")
    + w8 * F.col("Trash can percentile, 2022")
    + w9 * F.col("Premature mortality percentile, 2017")
    + w10 * F.col("Independency percentile, 2015-2019")
)

top_10_cds = overall_percentile.orderBy(F.col("Overall percentile").desc()).limit(10)
top_10_cds.show()

+---+--------------+-----+--------------------------+---------------------------------------------+---------------------+-----------------------------+------------------+-----------+----------------+----------------+--------------+-----------------------+---------+-----------------------------+------------------------------+--------------+----------------+-------------+--------------+-----------+------------------+--------------+----------------------------------+--------------------+--------------+---------------------------+------------------------+-------------------+-----------------------+-------------------+------------------------------------+----------------------------------+---------------------------------------+------------------------------+----------------------------+---------------------------------+------------------------------+------------------------------+-----------------------------+-------------------------+------------------------------------------+------------

3. Model training: train a machine learning model on the raw scores for the quality of life domains of each district, so that the model determines how the count of a facility type in a district may affect the raw scores of domains for that district.


4. Model execution: for each (district, facility type) pair, use the model to calculate the predicted domain scores, then calculate the predicted domain percentiles and the Predicted Overall Percentile Lift.


5. Civil planning: for each district, compare the Predicted Overall Percentile Lift of every facility type to determine which facility type would most improve the quality of life for that district, and for the city as a whole, compare the highest Predicted Overall Percentile Lift from each district to determine which facility types should be built and in which districts they should be built.


6. Interactive output: display both the input data and our calculated results on a user-friendly interactive map to aid in interpretation.